<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
👉 <b>Note:</b> This Jupyter notebook illustrates simplified usage of <code>rapidfireai</code>. For the full RapidFire AI experience with advanced experiment manager, UI, and production features, see our <a href=\"https://oss-docs.rapidfire.ai/en/latest/walkthrough.html\">Install and Get Started</a> guide.
<br/>
🎬 Watch our <a href=\"https://youtu.be/nPMBfZWqPWI\">intro video</a> to get started!\n
</div>


### Local Jupyter

This notebook is adapted to run in **Jupyter Lab / Notebook** on your machine. The Colab-oriented source is `rf-colab-*.ipynb` in the same folder (unchanged).

Use a Python 3.12+ environment with RapidFire installed (`pip install -e .` from the repo root, or `pip install rapidfireai`). Start services with `./setup/fit/start_dev.sh start` when the tutorial expects the dispatcher / MLflow / UI.


# RapidFire AI Tutorial: SFT with Trackio Experiment Tracking

This tutorial demonstrates how to fine-tune LLMs using **Supervised Fine-Tuning (SFT)** with [RapidFire AI](https://github.com/RapidFireAI/rapidfireai), enabling you to train and compare multiple configurations concurrently—even on a single GPU. We'll fine-tune a model on customer support data and explore how RapidFire AI's chunk-based scheduling delivers **16-24× faster experimentation throughput**.


## Install RapidFire AI Package and Services


In [ ]:
try:
    import rapidfireai
    print("✅ rapidfireai already installed")
except ImportError:
    %pip install rapidfireai==0.15.1 # Takes 1 min
    !rapidfireai init # Takes 1 min


## Start RapidFire Services

- If any issues arise, you can check the status in a terminal window using `rapidfireai status` or `rapidfireai doctor`.
- **Note:** You can also run `rapidfireai start` from a terminal in this notebook instead of the cell below.


In [ ]:
import subprocess
from time import sleep
import socket
try:
  s = [socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM)]
  s[0].connect(("127.0.0.1", 8851))
  s[1].connect(("127.0.0.1", 8852))
  s[2].connect(("127.0.0.1", 8853))
  s[0].close()
  s[1].close()
  s[2].close()
  print("RapidFire Services are running")
except OSError as error:
  print("RapidFire Services are not running, launching now...")
  subprocess.Popen(["rapidfireai", "start"])
  sleep(30)


## Configure RapidFire to Use Trackio


In [ ]:
import os

# Enable Trackio as the tracking backend
os.environ['RF_TRACKIO_ENABLED'] = 'true'


## Import RapidFire Components


In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

# NB: If you get "AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'" if you see protobuf issues, rerun this cell


## Load Dataset


In [ ]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# REDUCED dataset for memory constraints in this environment
train_dataset = dataset["train"].select(range(64))  # Reduced from 128
eval_dataset = dataset["train"].select(range(50, 60))  # 10 examples
train_dataset = train_dataset.shuffle(seed=42)
eval_dataset = eval_dataset.shuffle(seed=42)


## Define Data Processing Function

We'll format the data as Q&A pairs for fine-tuning:


In [ ]:
def sample_formatting_function(example):
    """Format the dataset for fine-tuning"""
    return {
        "text": f"Question: {example['instruction']}\nAnswer: {example['response']}"
    }

# Apply formatting to datasets
eval_dataset = eval_dataset.map(sample_formatting_function)
train_dataset = train_dataset.map(sample_formatting_function)


## Define Metrics Function

We'll use a lightweight metrics computation with just ROUGE-L to save memory:


In [ ]:
def sample_compute_metrics(eval_preds):
    """Lightweight metrics computation"""
    predictions, labels = eval_preds

    try:
        import evaluate

        # Only compute ROUGE-L (skip BLEU to save memory)
        rouge = evaluate.load("rouge")
        rouge_output = rouge.compute(
            predictions=predictions,
            references=labels,
            use_stemmer=True,
            rouge_types=["rougeL"]  # Only compute rougeL
        )

        return {
            "rougeL": round(rouge_output["rougeL"], 4),
        }
    except Exception as e:
        # Fallback if metrics fail
        print(f"Metrics computation failed: {e}")
        return {}


## Initialize Experiment


In [ ]:
# Create experiment with unique name
experiment = Experiment(experiment_name="trackio-demo-1")


## Define Model Configurations

This tutorial showcases Qwen3-0.6B (600M parameters), a modern and efficient model with Apache 2.0 license:


In [ ]:
# Qwen3 LoRA configs - standard transformer module names
peft_configs_lite = List([
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],  # Query and Value projections
        bias="none"
    ),
    RFLoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # All attention projections
        bias="none"
    )
])

# 2 configs with Qwen3-0.6B (modern, efficient model)
config_set_lite = List([
    RFModelConfig(
        model_name="Qwen/Qwen3-0.6B",  # 0.6B params, Apache 2.0 license
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=5e-4,  # Low lr for more stability
            lr_scheduler_type="linear",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,  # Effective bs = 4
            max_steps=64, # Raise this to see more learning
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            per_device_eval_batch_size=2,
            fp16=True,
            gradient_checkpointing=True,  # Save memory
            report_to="none",  # Disables wandb
        ),
        model_type="causal_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "float16",
            "use_cache": False
        },
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 128,
            "temperature": 0.7,
            "top_p": 0.9,
            "top_k": 40,
            "repetition_penalty": 1.1,
        }
    ),
    RFModelConfig(
        model_name="Qwen/Qwen3-0.6B",
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=2e-4,  # Even more conservative
            lr_scheduler_type="cosine",  # Try cosine schedule
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,
            max_steps=64, # Raise this to see more learning behaviors
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            per_device_eval_batch_size=2,
            fp16=True,
            gradient_checkpointing=True,
            report_to="none",  # Disables wandb
            warmup_steps=10,  # Add warmup for stability
        ),
        model_type="causal_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "float16",
            "use_cache": False
        },
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 128,
            "temperature": 0.7,
            "top_p": 0.9,
            "top_k": 40,
            "repetition_penalty": 1.1,
        }
    )
])


In [ ]:
def sample_create_model(model_config):
    """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_type = model_config["model_type"]
    model_kwargs = model_config["model_kwargs"]

    if model_type == "causal_lm":
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    else:
        raise ValueError(f"Unknown model_type: {model_type}. Supported types: 'causal_lm'")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Set pad token if not defined (common for decoder-only models)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id

    # Left padding is required for decoder-only causal LMs during batched generation
    tokenizer.padding_side = "left"

    return (model, tokenizer)


In [ ]:
# Simple grid search across all config combinations: 4 total (2 LoRA configs × 2 trainer configs)
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type="SFT"
)


## View Metrics with Trackio Dashboard

**Run this cell to display the Trackio dashboard. Metrics will appear in real-time as training progresses.**


In [ ]:
# Display the Trackio dashboard
import trackio
trackio.show(project=experiment.experiment_name)


## Run Training + Validation

Now we get to the main function for running multi-config training and evals. The metrics will appear in the Trackio dashboard above in real-time.


In [ ]:
# Launch train and validation for all configs in the config_group with swap granularity of 4 chunks for hyperparallel execution
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    eval_dataset,
    num_chunks=4,
    seed=42
)


## Launch Interactive Run Controller

RapidFire AI provides an Interactive Controller that lets you manage executing runs dynamically in real-time from the notebook:

- ⏹️ **Stop**: Gracefully stop a running config
- ▶️ **Resume**: Resume a stopped run
- 🗑️ **Delete**: Remove a run from this experiment
- 📋 **Clone**: Create a new run by editing the config dictionary of a parent run to try new knob values; optional warm start of parameters
- 🔄 **Refresh**: Update run status and metrics

The Controller uses ipywidgets and works in Jupyter Lab / Notebook (ipywidgets 8.x); Google Colab is also supported (ipywidgets 7.x).


In [ ]:
# Create Interactive Controller
sleep(15)
from rapidfireai.fit.utils.interactive_controller import InteractiveController

controller = InteractiveController(dispatcher_url="http://127.0.0.1:8851")
controller.display()


## End Experiment


In [ ]:
import ipywidgets as widgets
from IPython.display import display


def _on_end_clicked(_b):
    experiment.end()
    print("Done!")
    end_btn.disabled = True
    end_btn.description = "Experiment ended"


end_btn = widgets.Button(description="Click to End Experiment", button_style="info")
end_btn.on_click(_on_end_clicked)
display(end_btn)


## View RapidFire AI Log Files

You can track the work being done by the system via the RapidFire AI-produced log files in the logs/experiments/ folder.


In [ ]:
# Get the experiment-specific log file
from IPython.display import display, Pretty
log_file = experiment.get_log_file_path()

display(Pretty(f"📄 Experiment Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))


In [ ]:
# Get the training-specific log file
log_file = experiment.get_log_file_path("training")

display(Pretty(f"📄 Training Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))
